In [1]:
import pandas as pd

from krfp_models import krfp_models
from mpnn.molecule_o_variants import LeafActivatedMoleculeODetector, ActivationMovingMoleculeODetector

In [2]:
leaf_activated = pd.read_csv(
    "../artifacts/atypical_models_explainability_method_tester/R2-2026-07-24-19-20-15/LeafActivatedMoleculeODetector/positive_explainability_results.csv")

activation_moving = pd.read_csv(
    "../artifacts/atypical_models_explainability_method_tester/R2-2026-07-24-19-20-15/ActivationMovingMoleculeODetector/positive_explainability_results.csv"
)

smoothing = pd.read_csv(
    "../artifacts/atypical_models_explainability_method_tester/R2-2026-07-25-11-46-36/SmoothingMoleculeODetector/positive_explainability_results.csv"
)

method_names = [
    "PG Explainer",
    "Integrated Gradients",
    "Saliency",
    "SHAP Sampling",
    "Input x Gradient",
    "GNN Explainer",
    "SubgraphX"
]

flucarb_labels = pd.read_csv(
    "../artifacts/FluCarbGINE/positive_explainability_results.csv"
).keys()[1:]


In [3]:
def get_stats(df, stat):
    auroc = df[stat].dropna().mean()
    ideal = (df[stat].dropna() == 1.0).sum()
    full = len(df[stat].dropna())

    ideal_rate = ideal / full

    return auroc, ideal, full, ideal_rate

In [4]:
print("LEAF ACTIVATED/PGE")

auroc, ideal, full, ideal_rate = get_stats(leaf_activated, "PG Explainer_auroc")
print("Mean AUROC:", auroc)
print("Non-ideal performance", f"{(full - ideal)} / {full}")
print("Ideal rate:", ideal_rate)

LEAF ACTIVATED/PGE
Mean AUROC: 1.0
Non-ideal performance 0 / 9999
Ideal rate: 1.0


In [5]:
print("ACTIVATION MOVING/PGE")

auroc, ideal, full, ideal_rate = get_stats(activation_moving, "PG Explainer_auroc")
print("Mean AUROC:", auroc)
print("Non-ideal performance", f"{(full - ideal)} / {full}")
print("Ideal rate:", ideal_rate)

ACTIVATION MOVING/PGE
Mean AUROC: 0.9832364322861443
Non-ideal performance 9499 / 9999
Ideal rate: 0.05000500050005


In [6]:
print("SMOOTHING/PGE")

auroc, ideal, full, ideal_rate = get_stats(smoothing, "PG Explainer_auroc")
print("Mean AUROC:", auroc)
print("Non-ideal performance", f"{(full - ideal)} / {full}")
print("Ideal rate:", ideal_rate)

SMOOTHING/PGE
Mean AUROC: 0.5
Non-ideal performance 9999 / 9999
Ideal rate: 0.0


In [7]:
print("SMOOTHING/IG")

auroc, ideal, full, ideal_rate = get_stats(smoothing, "Integrated Gradients_auroc")
print("Mean AUROC:", auroc)
print("Non-ideal performance", f"{(full - ideal)} / {full}")
print("Ideal rate:", ideal_rate)

SMOOTHING/IG
Mean AUROC: 0.965836568817824
Non-ideal performance 9239 / 9999
Ideal rate: 0.07600760076007601


In [14]:
def get_stats_flucarb(name):
    df = pd.read_csv(f"../artifacts/FluCarbGINE/{name}_positive_explainability_results.csv")

    stat = f"{name}_auroc"

    auroc = df[stat].dropna().mean()
    ideal = (df[stat].dropna() == 1.0).sum()
    full = len(df[stat].dropna())

    ideal_rate = ideal / full

    return auroc, ideal, full, ideal_rate


def get_negative_stats_flucarb(name):
    if "SubgraphX" in name:
        raise ValueError("We do not test SubgraphX on negative examples")
    else:
        df = pd.read_csv(
            f"../artifacts/FluCarbGINE/{name}_negative_explainability_results.csv")

    results = df[f"{name}_iqr_success"].dropna().mean()

    return results

In [15]:
for method in method_names:
    print(f"FLUCARB/{method}/AUROC")

    auroc, ideal, full, ideal_rate = get_stats_flucarb(method)
    print("Mean AUROC:", auroc)
    print("Non-ideal performance", f"{(full - ideal)} / {full}")
    print("Ideal rate:", ideal_rate)
    print("=====")

FLUCARB/PG Explainer/AUROC
Mean AUROC: 0.5
Non-ideal performance 9999 / 9999
Ideal rate: 0.0
=====
FLUCARB/Integrated Gradients/AUROC
Mean AUROC: 0.7077350608183017
Non-ideal performance 9999 / 9999
Ideal rate: 0.0
=====
FLUCARB/Saliency/AUROC
Mean AUROC: 0.7883972603235876
Non-ideal performance 9996 / 9999
Ideal rate: 0.00030003000300030005
=====
FLUCARB/SHAP Sampling/AUROC
Mean AUROC: 0.6493590954248591
Non-ideal performance 100 / 100
Ideal rate: 0.0
=====
FLUCARB/Input x Gradient/AUROC
Mean AUROC: 0.7077350608183017
Non-ideal performance 9999 / 9999
Ideal rate: 0.0
=====
FLUCARB/GNN Explainer/AUROC
Mean AUROC: 0.7075123004470264
Non-ideal performance 9999 / 9999
Ideal rate: 0.0
=====
FLUCARB/SubgraphX/AUROC
Mean AUROC: 0.656018168658276
Non-ideal performance 100 / 100
Ideal rate: 0.0
=====


In [16]:
for method in method_names:
    if method == "SubgraphX":
        continue
    results = get_negative_stats_flucarb(method)

    print(f"FLUCARB NEGATIVE/{method}")
    print("Mean Succes Rate:", results)


FLUCARB NEGATIVE/PG Explainer
Mean Succes Rate: 1.0
FLUCARB NEGATIVE/Integrated Gradients
Mean Succes Rate: 0.1478
FLUCARB NEGATIVE/Saliency
Mean Succes Rate: 0.059
FLUCARB NEGATIVE/SHAP Sampling
Mean Succes Rate: 0.36
FLUCARB NEGATIVE/Input x Gradient
Mean Succes Rate: 0.2274
FLUCARB NEGATIVE/GNN Explainer
Mean Succes Rate: 0.1516


In [17]:
for model, _, _ in krfp_models:
    for i in range(1, 10):
        layer_name = f"mpnn{i}"
        if hasattr(model, layer_name):
            layer = getattr(model, layer_name)
            bias = layer.bias

            positives_count = (bias > 0).sum().item()
            assert positives_count == 0, f"WOMBAT whiteboxes generally should not have positive biases"

In [18]:
from mpnn.molecule_o_variants import SmoothingMoleculeODetector

for model in [LeafActivatedMoleculeODetector(), SmoothingMoleculeODetector(), ActivationMovingMoleculeODetector()]:
    for i in range(1, 10):
        layer_name = f"mpnn{i}"
        if hasattr(model, layer_name):
            layer = getattr(model, layer_name)
            bias = layer.bias

            positives_count = (bias > 0).sum().item()

            print("Positive biases for", model.__class__.__name__, "layer", layer_name, ":", positives_count)

Positive biases for LeafActivatedMoleculeODetector layer mpnn1 : 0
Positive biases for LeafActivatedMoleculeODetector layer mpnn2 : 0
Positive biases for SmoothingMoleculeODetector layer mpnn1 : 0
Positive biases for SmoothingMoleculeODetector layer mpnn2 : 0
Positive biases for SmoothingMoleculeODetector layer mpnn3 : 0
Positive biases for SmoothingMoleculeODetector layer mpnn4 : 0
Positive biases for SmoothingMoleculeODetector layer mpnn5 : 0
Positive biases for SmoothingMoleculeODetector layer mpnn6 : 0
Positive biases for ActivationMovingMoleculeODetector layer mpnn1 : 0
Positive biases for ActivationMovingMoleculeODetector layer mpnn2 : 1
Positive biases for ActivationMovingMoleculeODetector layer mpnn3 : 0
